In [1]:
# Montar Google Drive (si ejecutamos en Google Colab) y definir directorio de datos
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/NLP/Practica/'
except ImportError:
    data_dir = './'

# Nos aseguramos de que el directorio exista
os.makedirs(data_dir, exist_ok=True)

# Cargamos los datos preprocesados
import pickle
import numpy as np
with open(os.path.join(data_dir, 'preprocessed_data.pkl'), 'rb') as f:
    x_train, x_test, y_train, y_test, tfidf, x_train_, x_test_, median_words_rw = pickle.load(f)

# Definimos la función bold para mostrar texto en negrita
def bold(text):
    return f"\033[1m{text}\033[0m"


Mounted at /content/drive


# 3 - Modelo de ML

## Train & Test

In [2]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import time

In [3]:
# Como método para equilibrar el desbalanceo de muestras (~88% positivas) durante el entrenamiento
# he encontrado la opción scale_pos_weight que sirve para balancear los pesos positivos y negativos
# https://medium.com/@mate.voros1998/handling-imbalanced-datasets-with-xgboost-optimizing-model-performance-with-smart-parameter-tuning-18568c7783cf
# Asesor: DeepSeek

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale = neg_count / pos_count  # ~0.136 in your case

xgb_model = XGBClassifier(n_estimators=750, learning_rate=0.05, scale_pos_weight=scale, eval_metric='logloss')

ts_start = int(time.time())
xgb_model.fit(x_train_, y_train, verbose=True)
ts_end = int(time.time())
xgb_train_time = ts_end - ts_start
print(bold('Time:'), xgb_train_time)

Time: 46


In [4]:
y_pred_xgb = xgb_model.predict(x_test_)

## Métricas y Conclusiones (ML)

In [5]:
from sklearn.metrics import accuracy_score
from matplotlib import pyplot


In [6]:

# evaluate predictions
y_pred_xgb = xgb_model.predict(x_test_)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
print(bold('Accuracy:')+' %.2f%%' % (xgb_accuracy * 100.0))


Accuracy: 78.92%


In [7]:
# Get feature names and their importance
feature_names = tfidf.get_feature_names_out()
importance = xgb_model.feature_importances_

# Pair them and sort
feature_importance = list(zip(feature_names, importance))
feature_importance_sorted = sorted(feature_importance, key=lambda x: x[1], reverse=True)

# top 10
top_10 = feature_importance_sorted[:10]
for feature, score in top_10:
    print(f"{feature}: {score:.4f}")

easy: 0.0133
great: 0.0092
ok: 0.0083
love: 0.0073
needed: 0.0068
best: 0.0066
perfectly: 0.0066
perfect: 0.0063
highly: 0.0062
awesome: 0.0060


In [8]:
# Métricas
def metricas_xgb():
  length_xgb = len(y_pred_xgb)
  count_0_xgb = 0
  count_1_xgb = 0
  for j in range(len(y_pred_xgb)):
    if y_pred_xgb[j] == 0:
      count_0_xgb += 1
    else:
      count_1_xgb += 1
  return count_0_xgb, count_1_xgb, length_xgb

def mostrar_metricas_xgb():
  count_0_xgb, count_1_xgb, length_xgb = metricas_xgb()
  print(bold('Prediction xgb'))
  print(' xgb accuracy: {:>4.2f}'.format(xgb_accuracy*100))
  print('        0 xgb: {:>4}'.format(count_0_xgb), '({:>6.2f}%)'.format(count_0_xgb/length_xgb * 100))
  print('        1 xgb: {:>4}'.format(count_1_xgb), '({:>6.2f}%)'.format(count_1_xgb/length_xgb * 100))

In [9]:
mostrar_metricas_xgb()

Prediction xgb
 xgb accuracy: 78.92
        0 xgb:  569 ( 22.17%)
        1 xgb: 1997 ( 77.83%)


# 4 - Modelo de DL

## Train & Test

In [10]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense

In [11]:
vocabulary_size = len(tfidf.vocabulary_)

In [12]:
# Create a Keras Sequential model
ff_model = Sequential([
  Dense(128, activation='elu'),                      # second hidden layer with 32 neuron
  Dense(64, activation='elu'),                      # second hidden layer with 32 neuron
  Dense(32, activation='elu'),                       # third hidden layer with 8 neuron
  Dense(1, activation='sigmoid')            #output layer
])
ff_model.compile(optimizer='adam', loss='binary_crossentropy')

In [13]:
ff_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
time_ff_ini = time.time()

hist = ff_model.fit(
    x_train_,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(x_train_, y_train),
    verbose=2
)
ff_train_time = time.time() - time_ff_ini


Epoch 1/20
241/241 - 5s - 19ms/step - loss: 0.3454 - val_loss: 0.2315
Epoch 2/20
241/241 - 2s - 9ms/step - loss: 0.2380 - val_loss: 0.1655
Epoch 3/20
241/241 - 2s - 9ms/step - loss: 0.1961 - val_loss: 0.1404
Epoch 4/20
241/241 - 6s - 25ms/step - loss: 0.1702 - val_loss: 0.1287
Epoch 5/20
241/241 - 2s - 10ms/step - loss: 0.1513 - val_loss: 0.1174
Epoch 6/20
241/241 - 2s - 9ms/step - loss: 0.1346 - val_loss: 0.0968
Epoch 7/20
241/241 - 3s - 12ms/step - loss: 0.1090 - val_loss: 0.0679
Epoch 8/20
241/241 - 2s - 10ms/step - loss: 0.0832 - val_loss: 0.0551
Epoch 9/20
241/241 - 4s - 16ms/step - loss: 0.0643 - val_loss: 0.0400
Epoch 10/20
241/241 - 4s - 15ms/step - loss: 0.0486 - val_loss: 0.0264
Epoch 11/20
241/241 - 2s - 10ms/step - loss: 0.0353 - val_loss: 0.0167
Epoch 12/20
241/241 - 2s - 9ms/step - loss: 0.0232 - val_loss: 0.0111
Epoch 13/20
241/241 - 3s - 12ms/step - loss: 0.0143 - val_loss: 0.0139
Epoch 14/20
241/241 - 4s - 16ms/step - loss: 0.0236 - val_loss: 0.0149
Epoch 15/20
241/241

In [15]:
print(bold('Train time:'), ff_train_time)

Train time: 62.22453474998474


In [16]:
# Prediction
y_pred_ff_raw = ff_model.predict(x_test_)
y_pred_ff = (y_pred_ff_raw > 0.5).astype(int).flatten()  # Convert to 0 or 1

81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [17]:
# Calculate accuracy

ff_accuracy = accuracy_score(y_test, y_pred_ff)
print(bold('Test Accuracy:')+' {:4.2f}%'.format(ff_accuracy*100))

Test Accuracy: 83.28%


## Métricas y Conclusiones (DL)

In [18]:
def metricas_ff():
  length_ff = len(y_pred_ff)
  count_0_ff = 0
  count_1_ff = 0

  for j in range(length_ff):
    if y_pred_ff[j] == 0:
      count_0_ff += 1
    else:
      count_1_ff += 1
  return count_0_ff, count_1_ff, length_ff


def mostrar_metricas_ff():
  count_0_ff, count_1_ff, length_ff = metricas_ff()

  # Printing predictions
  print(bold('Prediction ffi'))
  print('  ff accuracy: {:>4.2f}'.format(ff_accuracy*100))
  print('  0 ff : {:>4}'.format(count_0_ff), '({:>6.2f}%)'.format(count_0_ff/length_ff * 100))
  print('  1 ff : {:>4}'.format(count_1_ff), '({:>6.2f}%)'.format(count_1_ff/length_ff * 100))
  print('\n')

In [19]:
mostrar_metricas_ff()

Prediction ffi
  ff accuracy: 83.28
  0 ff :  309 ( 12.04%)
  1 ff : 2257 ( 87.96%)




In [20]:
import random
for i in range(5):
  j = random.randint(0, len(y_pred_ff)-1)
  print ('random index: ' + bold(str(j)))
  print('>',x_test[j][:150],'<')
  print('   ff  score: ', y_pred_ff[j])
  print('   xgb score: ', y_pred_xgb[j])

random index: 2489
> im enjoying actually bought keep amp floor also like puts amp angle pointing upwards supporting 50 pound amp easily hasnt tilted back fallen im glad b <
   ff  score:  1
   xgb score:  1
random index: 589
> ergonomic tuner functions well sturdy fits ur pocket count fender give u good product <
   ff  score:  1
   xgb score:  1
random index: 1835
> youre need guitar capo pretty good offering pretty cheap get shipping fairly fast ready use straight package <
   ff  score:  1
   xgb score:  0
random index: 1094
> bought first guitar 6 months ago chose pack picks amazons list buyers also bought fender 351 picksat time figured picks created equalsince ive tried v <
   ff  score:  1
   xgb score:  1
random index: 2188
> lets get things straight start love effects pedals love idea handcrafted boutique pedal owned ill also play massproduced boss ibanez stuff put foreign <
   ff  score:  1
   xgb score:  1


In [21]:
print(bold('Accuracy')+'\n--------')
print('   XGB : ', np.round(xgb_accuracy*100, 2), '%')
print('   FF  : ', np.round(ff_accuracy*100, 2),'%')
print(bold('Train time')+'\n----------')
print('   XGB : {:.2f}'.format( xgb_train_time,2 ) + ' seg.')
print('   FF  : {:.2f}'.format( np.round(ff_train_time, 2)) + ' seg.')


Accuracy
--------
   XGB :  78.92 %
   FF  :  83.28 %
Train time
----------
   XGB : 46.00 seg.
   FF  : 62.22 seg.


In [22]:
# Guardamos los resultados de ML y DL para compararlos en el notebook del GRU
import pickle
with open(os.path.join(data_dir, 'ml_dl_results.pkl'), 'wb') as f:
    pickle.dump((y_pred_xgb, xgb_accuracy, xgb_train_time, y_pred_ff, ff_accuracy, ff_train_time), f)
